# BFMN — Rassereno

Building of **behavioural forma mentis networks (BFMN)** for every sample, extraction of
their network features and export of the aggregated results.

In [1]:
import pandas as pd
import pickle

from config import RESSERENO_ROOT, KEYWORDS_RASSERENO, COMMON_CUES_RASSERENO, CUES_EXAM_ANX
from bfmn_building import (
    categorize_valence_multiple,
    pipeline_df_building,
    build_multiple_fmnt,
)

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

### Function defaults

`build_multiple_fmnt`:
- `flower="yes"` &nbsp;&rarr;&nbsp; compute the emotion z-scores and plot the emotional flowers (set `"no"` to skip them)
- `language="ita"` &nbsp;&rarr;&nbsp; language EmoAtlas works in (set `"eng"` for English)
- `plot_keywords=None` &nbsp;&rarr;&nbsp; draw the frame of every keyword (a list draws only those, `[]` none)
- `flower_keywords=None` &nbsp;&rarr;&nbsp; draw the flower of every keyword (the z-scores are computed for all of them anyway)
- `plot_dir=None` &nbsp;&rarr;&nbsp; save the plots in `out/analysis/bfmn/<file_path>` and `out/analysis/emo_flowers_bfmn/<file_path>` (a path replaces `<file_path>`)

`pipeline_df_building`:
- `filter="no"` &nbsp;&rarr;&nbsp; keep every association (set `"yes"` to drop idiosyncratic ones, i.e. associations made by a single participant)


## Files reading

In [3]:
def read_formatting(rel_path):
    return pd.read_csv(RESSERENO_ROOT / "out" / "formatting" / rel_path)


def read_split(rel_path):
    return pd.read_csv(RESSERENO_ROOT / "out" / "splitted_dfs" / rel_path)


all_qualtrics = read_formatting(
    "all_students_2025/associations_valence_task_qualtrics.csv"
)
high_school_south = all_qualtrics[
    all_qualtrics["participant_id"].str.startswith("high_school_south")
]

# anxiety-split files: sample name -> splitted_dfs subfolder (psychology: only the students whose
# cue set had all four target words; GPT-oss: the personas matched to those humans)
SPLIT_FOLDERS = {
    "high_school_2025": "high_school_2025",
    "physics_2025": "physics_2025",
    "psychology": "psychology_exam_anx_paper",
    "gpt_high_school": "high_school_gpt_oss",
    "gpt_physics": "physics_gpt_oss",
    "gpt_psychology": "psychology_gpt_oss",
}
# high- and low-anxiety groups, and the whole sample without the anxiety split
GROUPS = {"high_anx": "high_anx.csv", "low_anx": "low_anx.csv", "all": "all.csv"}


def common_cues_only(df):
    """Keep the answers to the cue words shared by every sample's cue set."""
    return df[df["cue_word"].isin(COMMON_CUES_RASSERENO)].reset_index(drop=True)


SPLITS = {
    f"{sample}_{group}": common_cues_only(read_split(f"{folder}/{file}"))
    for sample, folder in SPLIT_FOLDERS.items()
    for group, file in GROUPS.items()
}

# frames are extracted for the keywords that are among the common cues (as spelled in the networks)
KEYWORDS = [
    k for k in KEYWORDS_RASSERENO if k in {c.replace("à", "a") for c in COMMON_CUES_RASSERENO}
]
KEYWORDS


['matematica',
 'statistica',
 'fisica',
 'scienza',
 'esame',
 'voto',
 'scuola',
 'ansia',
 'benessere']

## Dataframe preparation

Every sample goes through the full `pipeline_df_building` (text cleaning &rarr; long format
&rarr; valence categorization). Every sample (human and GPT-oss) is prepared twice: **unfiltered** (network features,
emotional flowers, exports) and **filtered** (semantic-frame plots only, keeping associations
given by at least two participants for the same cue word).

Only the answers to the 17 cue words shared by all cue sets (`COMMON_CUES_RASSERENO`) are used.


In [4]:
# every sample (human and GPT-oss) is prepared both filtered (plots) and unfiltered (analysis),
# with the same rule for all
sample_dfs = {
    f"{name}_{version}": pipeline_df_building(df, filter=flag)
    for name, df in SPLITS.items()
    for version, flag in [("filtered", "yes"), ("unfiltered", "no")]
}


Number of unique words: 198
Number of unique words: 777
Number of unique words: 218
Number of unique words: 844
Number of unique words: 371
Number of unique words: 1314
Number of unique words: 90
Number of unique words: 557
Number of unique words: 96
Number of unique words: 447
Number of unique words: 211
Number of unique words: 871
Number of unique words: 435
Number of unique words: 1286
Number of unique words: 453
Number of unique words: 1288
Number of unique words: 715
Number of unique words: 1976
Number of unique words: 219
Number of unique words: 753
Number of unique words: 213
Number of unique words: 653
Number of unique words: 330
Number of unique words: 1153
Number of unique words: 131
Number of unique words: 442
Number of unique words: 125
Number of unique words: 418
Number of unique words: 204
Number of unique words: 672
Number of unique words: 315
Number of unique words: 1013
Number of unique words: 351
Number of unique words: 1064
Number of unique words: 530
Number of uniqu

In [5]:
df_high_school_south = pipeline_df_building(high_school_south)

Number of unique words: 1299


In [6]:
fmnts, features_keyword, features_tot = build_multiple_fmnt(
    df_high_school_south, ["matematica", "fisica"], RESSERENO_ROOT, "high_school_south"
)

Local clustering matematica: 0.05794556628621598
Degree matematica: 68
Counts of 'matematica' = {'positive': 12, 'negative': 5, 'neutral': 52}
Mode of 'matematica' = neutral


findfont: Failed to find font weight light, now using 400.
findfont: Failed to find font weight demibold, now using 700.


Local clustering fisica: 0.04773082942097027
Degree fisica: 72
Counts of 'fisica' = {'positive': 18, 'negative': 4, 'neutral': 51}
Mode of 'fisica' = neutral


## BFMN building

Each sample is described by `(name, dataframe, keywords, output subfolder, extra kwargs)` and
built through the same `build_multiple_fmnt` call. Results are collected in `bfmn_results`, keyed
by sample name, each holding the keyword formae mentis, the per-keyword features and the whole-network
features.

Plots for the exam–anxiety paper go to `out/analysis/bfmn/exam_anxiety_paper/<sample>_filtered/<group>/` and
are drawn only for `CUES_EXAM_ANX`:
- **filtered** networks: frame plots only (associations given by at least two participants);
- **unfiltered** networks: no frame plots; they give every number in the paper (features, valence aura,
  emotion z-scores, the edge lists for the Jaccard analysis) and the emotional flowers of `CUES_EXAM_ANX`,
  saved to `out/analysis/emo_flowers_bfmn/exam_anxiety_paper/<sample>_unfiltered/<group>/`.


In [7]:
# (name, dataframe, keywords, analysis label, extra build_multiple_fmnt kwargs)
# the analysis label ("{folder}_{version}/{group}") names the sample in the exported features and
# seeds its emotion z-scores; the plots go to the same path inside PAPER_PLOTS
PAPER_PLOTS = "exam_anxiety_paper"
OUTPUT_FOLDERS = {
    "high_school_2025": "liceo_2025",
    "physics_2025": "physics_students_2025",
    "psychology": "psychology_students",
    "gpt_high_school": "gpt_oss_high_school",
    "gpt_physics": "gpt_oss_physics",
    "gpt_psychology": "gpt_oss_psychology",
}
GROUP_FOLDERS = {"high_anx": "high_anxiety", "low_anx": "low_anxiety", "all": "all"}
PLOTS_BY_VERSION = {
    # frame plots of CUES_EXAM_ANX only, no flowers
    "filtered": {"flower": "no"},
    # every keyword for the analysis, no frame plots, flowers of CUES_EXAM_ANX only
    "unfiltered": {"flower": "yes", "plot_keywords": [], "flower_keywords": CUES_EXAM_ANX},
}
BFMN_BUILDS = [
    (
        f"{sample}_{group}_{version}",
        sample_dfs[f"{sample}_{group}_{version}"],
        CUES_EXAM_ANX if version == "filtered" else KEYWORDS,
        f"{folder}_{version}/{GROUP_FOLDERS[group]}",
        {
            **PLOTS_BY_VERSION[version],
            "plot_dir": f"{PAPER_PLOTS}/{folder}_{version}/{GROUP_FOLDERS[group]}",
        },
    )
    for sample, folder in OUTPUT_FOLDERS.items()
    for group in GROUPS
    for version in ["filtered", "unfiltered"]
]

bfmn_results = {}
for name, df, keywords, file_path, kwargs in BFMN_BUILDS:
    fmnts, features_keyword, features_tot = build_multiple_fmnt(
        df, keywords, RESSERENO_ROOT, file_path, **kwargs
    )
    bfmn_results[name] = {
        "fmnts": fmnts,
        "features_keyword": features_keyword,
        "features_tot": features_tot,
    }


Local clustering voto: 0.07142857142857142
Degree voto: 21
Counts of 'voto' = {'positive': 0, 'negative': 7, 'neutral': 15}
Mode of 'voto' = neutral
Local clustering esame: 0.0915032679738562
Degree esame: 18
Counts of 'esame' = {'positive': 0, 'negative': 9, 'neutral': 10}
Mode of 'esame' = neutral
Local clustering ansia: 0.09090909090909091
Degree ansia: 22
Counts of 'ansia' = {'positive': 1, 'negative': 12, 'neutral': 10}
Mode of 'ansia' = negative
Local clustering benessere: 0
Degree benessere: 22
Counts of 'benessere' = {'positive': 13, 'negative': 0, 'neutral': 10}
Mode of 'benessere' = positive
Local clustering matematica: 0.03028083028083028
Degree matematica: 91
Counts of 'matematica' = {'positive': 11, 'negative': 13, 'neutral': 68}
Mode of 'matematica' = neutral
Local clustering statistica: 0.021176470588235293
Degree statistica: 51
Counts of 'statistica' = {'positive': 3, 'negative': 9, 'neutral': 40}
Mode of 'statistica' = neutral
Local clustering fisica: 0.027633378932968

## BFMN exports

- `features_keyword_fmns.csv` / `features_tot_fmns.csv`: per-keyword and whole-network features for
  every sample (human samples in their **unfiltered** version).
- `all_bfmns.pkl`: the edge lists of every keyword forma mentis (human samples in their **unfiltered**
  version, so the Jaccard analysis uses the same networks as the features).

In [8]:
analysis_dir = RESSERENO_ROOT / "out" / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

# samples (and order) feeding the exported feature tables, all in their unfiltered version
FEATURE_EXPORT_ORDER = [
    f"{sample}_{group}_unfiltered"
    for sample in OUTPUT_FOLDERS
    for group in GROUPS
]

for feature_key, filename in [
    ("features_keyword", "features_keyword_fmns.csv"),
    ("features_tot", "features_tot_fmns.csv"),
]:
    combined = pd.concat(
        [bfmn_results[name][feature_key] for name in FEATURE_EXPORT_ORDER],
        ignore_index=True,
    )
    combined.to_csv(analysis_dir / filename, index=False)


In [9]:
# export key -> sample name (unfiltered networks, so the Jaccard analysis uses the same networks
# as the features)
EXPORT_NAMES = {
    "high_school_2025": "high_school",
    "physics_2025": "physics",
    "psychology": "psychology",
    "gpt_high_school": "gpt_oss_high_school",
    "gpt_physics": "gpt_oss_physics",
    "gpt_psychology": "gpt_oss_psychology",
}
BFMN_EXPORT = {
    f"{export}_{group}": f"{sample}_{group}_unfiltered"
    for sample, export in EXPORT_NAMES.items()
    for group in GROUPS
}

all_bfmns = {
    export_key: [fmnt.edges for fmnt in bfmn_results[name]["fmnts"]]
    for export_key, name in BFMN_EXPORT.items()
}

with open(analysis_dir / "all_bfmns.pkl", "wb") as f:
    pickle.dump(all_bfmns, f)
